# LC 239 — Sliding Window Maximum
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Sliding Window
**Pattern:** Monotonic Decreasing Deque

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Keep a deque of indices
in decreasing order of their values. The front is
always the index of the current window's maximum.
Smaller elements behind a new large element can
never be the maximum — discard them.
</div>

## Official Problem Statement

You are given an array of integers `nums`, there is
a sliding window of size `k` which is moving from
the very left of the array to the very right. You
can only see the `k` numbers in the window. Each
time the sliding window moves right by one position.

Return the max sliding window.

**Example 1:**
```
Input:  nums = [1,3,-1,-3,5,3,6,7], k = 3
Output: [3,3,5,5,6,7]
```
**Example 2:**
```
Input:  nums = [1], k = 1
Output: [1]
```

**Constraints:**
- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`
- `1 <= k <= nums.length`

## What This Is Actually Asking

Slide a window of fixed size k across the array
from left to right, one step at a time.
At each position, record the maximum value inside
the window.
Return the list of all those maximums.
The challenge: do it without rescanning the whole
window each time.

## Walk Through an Example by Hand

```
nums = [1, 3, -1, -3, 5, 3, 6, 7]   k=3
idx:    0  1   2   3  4  5  6  7

dq = []   result = []

i=0 num=1   dq=[]    -> append 0    dq=[0]   (window not full)
i=1 num=3   nums[0]=1<3 -> pop 0    dq=[1]   (window not full)
i=2 num=-1  -1<3 -> keep           dq=[1,2]  window full!
            front=1 -> nums[1]=3   result=[3]
i=3 num=-3  -3<-1 -> keep          dq=[1,2,3]
            front=1 still in window (1 >= 3-3+1=1)
            result=[3,3]
i=4 num=5   nums[3]=-3<5 pop 3
            nums[2]=-1<5 pop 2
            nums[1]=3<5  pop 1     dq=[4]
            front=4 -> nums[4]=5  result=[3,3,5]
i=5 num=3   3<5 -> keep           dq=[4,5]
            front=4 -> nums[4]=5  result=[3,3,5,5]
i=6 num=6   nums[5]=3<6 pop 5
            nums[4]=5<6 pop 4     dq=[6]
            result=[3,3,5,5,6]
i=7 num=7   nums[6]=6<7 pop 6     dq=[7]
            result=[3,3,5,5,6,7]

Answer: [3,3,5,5,6,7]
```

## The Picture

```
nums = [1,  3, -1, -3,  5,  3,  6,  7]

Think of the deque as a VIP queue.
Only useful candidates stay in the queue.

Rules:
  1. New element arrives at the BACK.
  2. Evict anyone at the back who is SMALLER
     (they can never be the max while I'm here).
  3. Evict the FRONT if its index is outside the window.
  4. Front = current window maximum.

deque always holds indices in DECREASING value order:

  Step: [1]      dq=[0]       max=—  (not full)
  Step: [1,3]    dq=[1]       max=—  (1 evicted, 3>1)
  Step: [3,-1,-3] dq=[1,2,3]  max=3
  Step: [3,-1,-3,5] 5>all -> dq=[4]  max=5

Each element enters and leaves the deque exactly once
-> amortized O(1) per element -> O(n) total.
```

## When To Use This Pattern

- When you see **rolling maximum (or minimum) over
  a fixed window**, think **monotonic deque**
- When you need O(1) max per step (not O(k) rescan),
  think **deque with decreasing values front-to-back**
- When a new element makes old smaller elements
  useless forever, think **pop from the back**
- When the front index falls outside the window,
  think **popleft to evict stale max**

## The Approach

Use a deque that stores array indices in decreasing
order of their values — the front always holds the
index of the window's current maximum.
For each new element, remove from the back all
indices whose values are smaller (they can never
win while this larger element is in the window).
Remove from the front any index that has fallen
outside the window boundary.
Once the window is full (i >= k-1), append the
value at the front to the result.

In [3]:
from collections import deque  # O(1) append and popleft
from typing import List        # type hints

In [4]:
def test_harness(func):
    tests = [
        # (nums, k, expected)
        ([1,3,-1,-3,5,3,6,7], 3, [3,3,5,5,6,7]),
        ([1],                 1, [1]),
        ([1,-1],              1, [1,-1]),
        ([9,11],              2, [11]),
        ([4,-2],              2, [4]),
        ([1,3,1,2,0,5],       3, [3,3,2,5]),
        ([2,1,5,3,6,4,8,9,2], 4, [5,6,6,8,9,9]),
        ([7,2,4],             2, [7,4]),
    ]

    passed = 0
    for i, (nums, k, expected) in enumerate(tests):
        result = func(nums[:], k)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"nums={nums} k={k} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [51]:
from collections import deque  # O(1) append and popleft
from typing import List        # type hints
def maxSlidingWindow(
    nums: List[int], k: int
) -> List[int]:
    """
    So we kick numbers from the right (top) which are smaller than the current number .. 
    esentially keeping the left most position 0 biggest
    If a new number walks in and it's bigger than something already on the list — 
    cross that thing off. It can never win while the new guy is in the window.
    """
    """
    Monotonic deque — indices stored in decreasing value order.
    Left end is always the current window maximum.
    Two eviction rules keep it honest:
      - Right: kick anything smaller than the new arrival (can never win)
      - Left:  kick anything that has aged outside the window
    """
    dq = deque()
    res = []
    for r, num in enumerate(nums):
        while dq and nums[dq[-1]] < num:
            dq.pop()
        dq.append(r)
        #Guard 1 ... is the left most still in my window 
        #least number in my window is r-k+1
        if dq[0] < r-k+1: dq.popleft()
        #Now we need to fill items in the res; only if we have filled at least the first window
        # r >= k-1 .. for example k = 3 .. r=2 this means the first window established.
        # you will append the left most item in the window as it is the largest surviving element
        # in this window
        if r >= k-1: res.append(nums[dq[0]])
    return res
print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3)) # [3,3,5,5,6,7]
print(maxSlidingWindow([1], 1))         # [1]
print(maxSlidingWindow([9,11], 2))      # [11]
print(maxSlidingWindow([4,-2], 2))      # [4]
test_harness(maxSlidingWindow)

[3, 3, 5, 5, 6, 7]
[1]
[11]
[4]
Test 1: PASSED | nums=[1, 3, -1, -3, 5, 3, 6, 7] k=3 | expected=[3, 3, 5, 5, 6, 7] | got=[3, 3, 5, 5, 6, 7]
Test 2: PASSED | nums=[1] k=1 | expected=[1] | got=[1]
Test 3: PASSED | nums=[1, -1] k=1 | expected=[1, -1] | got=[1, -1]
Test 4: PASSED | nums=[9, 11] k=2 | expected=[11] | got=[11]
Test 5: PASSED | nums=[4, -2] k=2 | expected=[4] | got=[4]
Test 6: PASSED | nums=[1, 3, 1, 2, 0, 5] k=3 | expected=[3, 3, 2, 5] | got=[3, 3, 2, 5]
Test 7: PASSED | nums=[2, 1, 5, 3, 6, 4, 8, 9, 2] k=4 | expected=[5, 6, 6, 8, 9, 9] | got=[5, 6, 6, 8, 9, 9]
Test 8: PASSED | nums=[7, 2, 4] k=2 | expected=[7, 4] | got=[7, 4]

8/8 tests passed


In [13]:
from collections import deque
def maxSlidingWindow(
    nums: List[int], k: int
) -> List[int]:
    """
    Return the max of each sliding window of size k.

    Monotonic deque stores indices in decreasing value
    order. For each element: pop back if smaller than
    new value (useless), pop front if out of window.
    Append new index. Once window is full (i>=k-1),
    record nums[dq[0]] as the current max.

    Time:  O(n) — each index enters/exits deque once
    Space: O(k) — deque holds at most k indices
    """



# Quick debug — run this cell while building
print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))
# [3,3,5,5,6,7]
print(maxSlidingWindow([1], 1))         # [1]
print(maxSlidingWindow([9,11], 2))      # [11]
print(maxSlidingWindow([4,-2], 2))      # [4]

deque([[1, 0]])
deque([[1, 0], [3, 1]])
deque([[1, 0], [3, 1]])
deque([[1, 0], [3, 1]])
deque([[1, 0], [3, 1], [5, 4]])
deque([[1, 0], [3, 1], [5, 4]])
deque([[1, 0], [3, 1], [5, 4], [6, 6]])
deque([[1, 0], [3, 1], [5, 4], [6, 6], [7, 7]])
None
deque([[1, 0]])
None
deque([[9, 0]])
deque([[9, 0], [11, 1]])
None
deque([[4, 0]])
deque([[4, 0]])
None


In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxSlidingWindow)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — rescan every window | O(n × k) | O(1) |
| Segment tree / sparse table | O(n log n) | O(n) |
| Monotonic deque | O(n) | O(k) |

The deque achieves O(n) because each element is
added and removed at most once — the inner while
loops are amortized O(1) per element.

## Real World Connection

At Citi, the capacity planning dashboard shows the
rolling peak CPU over every 5-minute window across
6,000 servers — exactly the sliding window maximum
problem applied to telemetry streams.
Running a rescan of every window (O(n × k)) on
millions of data points per collection cycle would
exceed the latency budget; the monotonic deque
computes the same result in O(n).
On AWS, the Lambda function that publishes CloudWatch
custom metrics for peak-memory-per-window uses the
same deque pattern to avoid re-reading the full
sliding buffer on every tick.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra

In [50]:
from collections import deque
def maxSlidingWindow(
    nums: List[int], k: int
) -> List[int]:
    """
    Return the max of each sliding window of size k.

    Monotonic deque stores indices in decreasing value
    order. For each element: pop back if smaller than
    new value (useless), pop front if out of window.
    Append new index. Once window is full (i>=k-1),
    record nums[dq[0]] as the current max.

    Time:  O(n) — each index enters/exits deque once
    Space: O(k) — deque holds at most k indices
    """
    dq = deque()                               #indexes
    res = []
    for r, num in enumerate (nums, start=0):
        while dq and  nums[dq[-1]] < num:
            dq.pop()
        dq.append(r)
        if dq[0] < r-k+1: 
            dq.popleft() 
        if r >= k - 1:                #only if you are in a window meaning r > 2  if window of 3   
            res.append(dq[0])
    return([nums[i] for i in res])
    
# Quick debug — run this cell while building
print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3)) # [3,3,5,5,6,7]
print(maxSlidingWindow([1], 1))         # [1]
print(maxSlidingWindow([9,11], 2))      # [11]
print(maxSlidingWindow([4,-2], 2))      # [4]
test_harness(maxSlidingWindow)

[3, 3, 5, 5, 6, 7]
[1]
[11]
[4]
Test 1: PASSED | nums=[1, 3, -1, -3, 5, 3, 6, 7] k=3 | expected=[3, 3, 5, 5, 6, 7] | got=[3, 3, 5, 5, 6, 7]
Test 2: PASSED | nums=[1] k=1 | expected=[1] | got=[1]
Test 3: PASSED | nums=[1, -1] k=1 | expected=[1, -1] | got=[1, -1]
Test 4: PASSED | nums=[9, 11] k=2 | expected=[11] | got=[11]
Test 5: PASSED | nums=[4, -2] k=2 | expected=[4] | got=[4]
Test 6: PASSED | nums=[1, 3, 1, 2, 0, 5] k=3 | expected=[3, 3, 2, 5] | got=[3, 3, 2, 5]
Test 7: PASSED | nums=[2, 1, 5, 3, 6, 4, 8, 9, 2] k=4 | expected=[5, 6, 6, 8, 9, 9] | got=[5, 6, 6, 8, 9, 9]
Test 8: PASSED | nums=[7, 2, 4] k=2 | expected=[7, 4] | got=[7, 4]

8/8 tests passed


In [46]:
from Collections import deque
def maxSlidingWindow(
    nums: List[int], k: int
) -> List[int]:
    """
    Return the max of each sliding window of size k.

    Monotonic deque stores indices in decreasing value
    order. For each element: pop back if smaller than
    new value (useless), pop front if out of window.
    Append new index. Once window is full (i>=k-1),
    record nums[dq[0]] as the current max.

    Time:  O(n) — each index enters/exits deque once
    Space: O(k) — deque holds at most k indices
    """
    r"""
    monotonic increasing queue
    """
    if k > len (nums): return []
    dq, res = deque()  , []                    #insert only indexes that only if it represents a bigger number
    for r, num in enumerate(nums, start = 0):
        while dq and  nums[dq[-1]] < num:
            dq.pop()
        dq.append(r)
        #if index of the first item is outside the window ... remove the oldest item
        if dq[0] < r-k+1: 
            dq.popleft()
        if r >= k - 1:                #only if you are in a window meaning r > 2  if window of 3   
            res.append(nums[dq[0]])
    return res



# Quick debug — run this cell while building
print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))
# [3,3,5,5,6,7]
print(maxSlidingWindow([1], 1))         # [1]
print(maxSlidingWindow([9,11], 2))      # [11]
print(maxSlidingWindow([4,-2], 2))      # [4]
test_harness(maxSlidingWindow)

ModuleNotFoundError: No module named 'Collections'